# 🏗️ Notebook 1: Gmail — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/gmail
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A web-based email service: receive, store, search, and send email — at billions of messages.

### Functional requirements
- Send email via **SMTP** (to other providers).
- Receive email via **SMTP** (others sending to us).
- Web UI + IMAP for clients.
- **Search** across the mailbox (full-text).
- Labels/folders, threading, attachments.

### Non-functional
- Average mailbox: ~15 GB. 1B users → ~15 EB of storage.
- Read-heavy; inbox reads dominate writes.
- Spam filtering is mandatory.
- Attachments (up to 25 MB) need separate object storage.

### Protocols cheatsheet
- **SMTP** — send / receive between mail servers (port 25 server-to-server, 587 submission).
- **IMAP** — client reads server-side mailbox (port 993 TLS).
- **POP3** — older; client downloads and deletes (rarely used now).


## Architecture

```
    inbound sender ──▶ MX DNS ──▶ ┌────────────┐
                                   │ SMTP inbound│
                                   │ servers     │
                                   └────┬────────┘
                                        │
                                        ▼
                                ┌────────────────┐
                                │ Spam/virus     │
                                │ classifier     │
                                └────┬───────────┘
                                     │ clean → store
                                     ▼
  ┌──────────┐   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
  │ IMAP/web │◀─▶│ Mailbox API  │◀─▶│ Metadata DB  │   │ Object Store │
  │ (read)   │   │ (messages,   │   │ (Cassandra / │   │ (bodies +    │
  │          │   │  threads)    │   │  Bigtable)   │   │  attachments)│
  └──────────┘   └──────┬───────┘   └──────────────┘   └──────────────┘
                        │                                     ▲
                        │ writes                              │
                        ▼                                     │
                 ┌──────────────┐                             │
                 │ Search Index │ ← indexer pulls new msgs ───┘
                 │ (Lucene/ES)  │
                 └──────────────┘

  Outbound:
    client → SMTP submission (587) → our SMTP outbound → recipient MX
```

### Where's the scaling?
- **Metadata** is *hot* (list inbox, count unread). Partitioned by user_id, per-user ordering by date.
- **Message bodies** are *cold* after a few weeks. Move old ones to cheaper object storage.
- **Search index** is big; compressed + per-user shards.
